<a href="https://www.kaggle.com/code/eshitasriva/notebookf289bf7d9d?scriptVersionId=307111154" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kmldas/loan-default-prediction/Default_Fin.csv


# SECTION1: Classification By a Baseline model

In [2]:
df=pd.read_csv("/kaggle/input/datasets/kmldas/loan-default-prediction/Default_Fin.csv")
print(df.head())
print(df.columns)
print(df.info)
df["Defaulted?"].value_counts(normalize=True)
print("This Is An Imbalanced Dataset")

   Index  Employed  Bank Balance  Annual Salary  Defaulted?
0      1         1       8754.36      532339.56           0
1      2         0       9806.16      145273.56           0
2      3         1      12882.60      381205.68           0
3      4         1       6351.00      428453.88           0
4      5         1       9427.92      461562.00           0
Index(['Index', 'Employed', 'Bank Balance', 'Annual Salary', 'Defaulted?'], dtype='object')
<bound method DataFrame.info of       Index  Employed  Bank Balance  Annual Salary  Defaulted?
0         1         1       8754.36      532339.56           0
1         2         0       9806.16      145273.56           0
2         3         1      12882.60      381205.68           0
3         4         1       6351.00      428453.88           0
4         5         1       9427.92      461562.00           0
...     ...       ...           ...            ...         ...
9995   9996         1       8538.72      635908.56           0
9996   9997 

**IMPORTS**

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report



A confusion matrix shows the number of correct and incorrect predictions made by a classification model, while a classification report summarizes key evaluation metrics such as precision, recall, and F1-score.

In [4]:
X=df.drop(["Defaulted?","Index"],axis=1)
y=df["Defaulted?"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2, random_state=42)
model1 = LogisticRegression(max_iter=1000)
model1.fit(X_train,y_train)
preds1=model1.predict(X_test)

In [5]:
print(confusion_matrix(y_test, preds1))
print(classification_report(y_test, preds1))

[[1920   11]
 [  50   19]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1931
           1       0.63      0.28      0.38        69

    accuracy                           0.97      2000
   macro avg       0.80      0.63      0.68      2000
weighted avg       0.96      0.97      0.96      2000



The model achieved high accuracy but failed to detect most default cases, resulting in low recall. This highlights the impact of class imbalance and the need for better handling of minority classes.

# SECTION2: Balanced Logistic Regression

In [6]:
model2 = LogisticRegression(class_weight="balanced", max_iter=1000)
model2.fit(X_train,y_train)
preds2=model2.predict(X_test)
print(confusion_matrix(y_test,preds2))
print(classification_report(y_test,preds2))

[[1675  256]
 [   9   60]]
              precision    recall  f1-score   support

           0       0.99      0.87      0.93      1931
           1       0.19      0.87      0.31        69

    accuracy                           0.87      2000
   macro avg       0.59      0.87      0.62      2000
weighted avg       0.97      0.87      0.91      2000



In [7]:
results = []

for name, preds in [("Normal", preds1), ("Balanced", preds2)]:
    results.append([
        name,
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds)
    ])

pd.DataFrame(results, columns=["Model", "Precision", "Recall", "F1"])

,Model,Precision,Recall,F1
0,Normal,0.633333,0.275362,0.383838
1,Balanced,0.189873,0.869565,0.311688


Standard Model:
Misses most defaulters (high False Negatives, low True Positives), resulting in low recall.

Balanced Model:
More aggressive in predicting defaults (high False Positives), resulting in high recall but low precision.

# SECTION3: RandomForest and XGBoost

In [8]:
from sklearn.ensemble import RandomForestClassifier
rf= RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train,y_train)
rf_preds = rf.predict(X_test)

print("Random Forest Results")
print(confusion_matrix(y_test, rf_preds))
print(classification_report(y_test, rf_preds))

Random Forest Results
[[1750  181]
 [  17   52]]
              precision    recall  f1-score   support

           0       0.99      0.91      0.95      1931
           1       0.22      0.75      0.34        69

    accuracy                           0.90      2000
   macro avg       0.61      0.83      0.65      2000
weighted avg       0.96      0.90      0.93      2000



In [9]:
from xgboost import XGBClassifier

xgb= XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample= 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = len(y_train[y_train==0])/len(y_train[y_train==1]),
    random_state=42
)

xgb.fit(X_train,y_train)
xgb_preds = xgb.predict(X_test)

print("XGBoost Results")
print(confusion_matrix(y_test, xgb_preds))
print(classification_report(y_test, xgb_preds
                           ))

XGBoost Results
[[1743  188]
 [  13   56]]
              precision    recall  f1-score   support

           0       0.99      0.90      0.95      1931
           1       0.23      0.81      0.36        69

    accuracy                           0.90      2000
   macro avg       0.61      0.86      0.65      2000
weighted avg       0.97      0.90      0.93      2000



The Random Forest model achieved higher recall, making it more effective at identifying defaulters, but at the cost of increased false positives. In contrast, XGBoost provided a more balanced performance, improving precision and overall accuracy while slightly reducing recall. This demonstrates the trade-off between maximizing risk detection and minimizing false alarms, with model choice depending on business requirements.